In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DataType

In [2]:
spark = SparkSession.builder \
	.master("local[*]") \
	.appName("test") \
	.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 21:10:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.option("header","true").csv("output/emp")
df.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary|hired_date|
+-----------+-------------+-------------+---+------+-------+----------+
|         17|          105|  George Wang| 34|  Male|57000.0|2016-03-15|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|         11|          104|   David Park| 38|  Male|65000.0|2015-11-01|
|         12|          105|   Susan Chen| 31|Female|54000.0|2017-02-15|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|         15|          106|  Michael Lee| 37|  Male|63000.0|2014

In [5]:
# Case when 

emp = df.withColumn("new_gender", expr("case when gender = 'Male' then 'M' when gender = 'Female' then 'F' else null end"))

emp.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+
|employee_id|department_id|         name|age|gender| salary|hired_date|new_gender|
+-----------+-------------+-------------+---+------+-------+----------+----------+
|         17|          105|  George Wang| 34|  Male|57000.0|2016-03-15|         M|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|         M|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|         F|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|         M|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|         F|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|         M|
|         11|          104|   David Park| 38|  Male|65000.0|2015-11-01|         M|
|         12|          105|   Susan Chen| 31|Female|54000.0|2017-02-15|         F|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|         M|
|   

In [7]:
# Replace for string data type
emp_new_name = emp.withColumn("new_name", regexp_replace(col("name"),'J','Z'))
emp_new_name.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+-------------+
|employee_id|department_id|         name|age|gender| salary|hired_date|new_gender|     new_name|
+-----------+-------------+-------------+---+------+-------+----------+----------+-------------+
|         17|          105|  George Wang| 34|  Male|57000.0|2016-03-15|         M|  George Wang|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|         M|  Steven Chen|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|         F|    Grace Kim|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|         M|Zames Zohnson|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|         F|     Kate Kim|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|         M|      Tom Tan|
|         11|          104|   David Park| 38|  Male|65000.0|2015-11-01|         M|   David Park|
|         12|          105|   

In [8]:
emp_new_name.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hired_date: string (nullable = true)
 |-- new_gender: string (nullable = true)
 |-- new_name: string (nullable = true)



In [11]:
emp_date_fix = emp_new_name.withColumn("hired_date", to_date(col("hired_date"), 'yyyy-MM-dd'))
emp_date_fix.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hired_date: date (nullable = true)
 |-- new_gender: string (nullable = true)
 |-- new_name: string (nullable = true)



In [12]:
emp_current_date = emp_date_fix.withColumn("current_date", current_date()).withColumn("current_ts", current_timestamp())
emp_current_date.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+-------------+------------+--------------------+
|employee_id|department_id|         name|age|gender| salary|hired_date|new_gender|     new_name|current_date|          current_ts|
+-----------+-------------+-------------+---+------+-------+----------+----------+-------------+------------+--------------------+
|         17|          105|  George Wang| 34|  Male|57000.0|2016-03-15|         M|  George Wang|  2026-04-14|2026-04-14 21:30:...|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|         M|  Steven Chen|  2026-04-14|2026-04-14 21:30:...|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|         F|    Grace Kim|  2026-04-14|2026-04-14 21:30:...|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|         M|Zames Zohnson|  2026-04-14|2026-04-14 21:30:...|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|         F| 

In [15]:
# Handle null values
emp_null_df = emp_date_fix.withColumn("new_gender", coalesce(col("new_gender"),lit("O")))
emp_null_df.show(50,truncate=False)

+-----------+-------------+-------------+---+------+-------+----------+----------+-------------+
|employee_id|department_id|name         |age|gender|salary |hired_date|new_gender|new_name     |
+-----------+-------------+-------------+---+------+-------+----------+----------+-------------+
|17         |105          |George Wang  |34 |Male  |57000.0|2016-03-15|M         |George Wang  |
|19         |103          |Steven Chen  |36 |Male  |62000.0|2015-08-01|M         |Steven Chen  |
|20         |102          |Grace Kim    |32 |Female|53000.0|2018-11-01|F         |Grace Kim    |
|7          |101          |James Johnson|42 |Male  |70000.0|2012-03-15|M         |Zames Zohnson|
|8          |102          |Kate Kim     |29 |Female|51000.0|2019-10-01|F         |Kate Kim     |
|9          |103          |Tom Tan      |33 |Male  |58000.0|2016-06-01|M         |Tom Tan      |
|11         |104          |David Park   |38 |Male  |65000.0|2015-11-01|M         |David Park   |
|12         |105          |Sus

In [16]:
# Format date time to date string
emp_date_string = emp_date_fix.withColumn("date_string", date_format(col("hired_date"), "MM/dd/yyyy"))
emp_date_string.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+-------------+-----------+
|employee_id|department_id|         name|age|gender| salary|hired_date|new_gender|     new_name|date_string|
+-----------+-------------+-------------+---+------+-------+----------+----------+-------------+-----------+
|         17|          105|  George Wang| 34|  Male|57000.0|2016-03-15|         M|  George Wang| 03/15/2016|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|         M|  Steven Chen| 08/01/2015|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|         F|    Grace Kim| 11/01/2018|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|         M|Zames Zohnson| 03/15/2012|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|         F|     Kate Kim| 10/01/2019|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|         M|      Tom Tan| 06/01/2016|
|         11|      